# Track B — YOLOv8n 416×416 → ONNX (opset 11)

**Proyecto:** `embebidos-3` — clasificador de residuos en Jetson Nano B01
**Plataforma:** Kaggle T4 (preferido por sesión de 12 h + cuota 30 h/sem) o Colab T4.
**Pre-requisito:** Roboflow Version 1-B generada con `Fit (black edges) in 416×416`, export `yolov8`.
**Salida:** `yolov8n_waste.onnx` (opset 11) → compilar a `.engine` FP16 en el propio Jetson Nano con `trtexec --fp16 --workspace=1024`.

---

## Stack training vs runtime (mayo 2026)

| | Training (Kaggle/Colab) | Runtime (Jetson Nano JetPack 4.6.1) |
|---|---|---|
| Python | 3.10+/3.12 | 3.6.9 |
| PyTorch | 2.9-2.10 preinstalado | — (ONNX runtime via TRT) |
| Ultralytics | ≥8.4.46,<8.5 | — |
| ONNX opset | 11 forzado (default 20) | máx 13 leíble |
| onnxsim | ≥0.6.2,<0.7 | — |
| CUDA | 12.x host (irrelevante para `.onnx`) | 10.2 |
| TRT | — | 8.2.1 |
| Hardware | T4 GPU | Maxwell 128 cores **sin Tensor Cores INT8** |

Decisión completa: ver `investigaciones/2026-05-12/2026-05-12-compatibilidad-notebooks-training.md`.

## Decisiones críticas

| Decisión | Razón |
|---|---|
| `imgsz=416` (no 640) | Nature Sci Rep 2024 (DOI 10.1038/s41598-024-74798-3 Tabla 4): +25% FPS en Jetson Nano vs 640 sin pérdida importante de mAP. |
| TensorRT FP16 (no INT8) | Maxwell carece de tensor cores INT8. Qengineering: *"INT8 no aumenta FPS y degrada mAP significativamente"* en Nano. |
| Mosaic + Mixup ON (defaults) | Crasto 2024: mosaic +8.0 pp mAP50, +mixup +11.3 pp en YOLOv5 single-stage con desbalance foreground-foreground (idéntico a este dataset). |
| NO class weights | Crasto 2024: -1.9 pp en single-stage; Ultralytics #20259: incompatible mecánicamente con mosaic. |
| **`opset=11` forzado** | TRT 8.0/8.2 del JetPack 4.6.x no lee opset > 13. Ultralytics 8.4.x con torch 2.9+ usa opset 20 por default; hay que pasarlo explícito. |
| NMS en CPU NumPy en Nano | `EfficientNMS_TRT` plugin roto en Maxwell (NVIDIA/TensorRT #1538). |
| Ultralytics ≥ 8.4.46 | PR #24028 fix INT8 calibration con `imgsz` no-cuadrado; v8.4.48 (8 mayo 2026) última estable. |
| **NO `DATASET_DIRECTORY` env var** | Entra en conflicto con `location` arg del SDK Roboflow: genera dos ubicaciones distintas y deja ambas vacías (observado en Kaggle 2026-05-12). Mitigación real: chdir + búsqueda exhaustiva con yaml parse + unzip defensivo (cell-10). |


## 0. Detección de entorno y pre-flight

Auto-detecta Colab/Kaggle/local, valida GPU/RAM/disco, recomienda el batch size apropiado.

In [ ]:
import os, sys, platform, shutil, subprocess, time, json
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)
PLATFORM = "Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Local"

def banner(title: str) -> None:
    print("\n" + "=" * 64); print(f"  {title}"); print("=" * 64)

def t_ts() -> str:
    return time.strftime("%H:%M:%S")

banner("Track B · YOLOv8n 416 -> ONNX opset 11")
print(f"[{t_ts()}] Plataforma : {PLATFORM}")
print(f"[{t_ts()}] Python     : {sys.version.split()[0]}")

gpu_info, gpu_mem_total_mb = "", 0
try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, timeout=5,
    )
    parts = [p.strip() for p in out.stdout.strip().split(",")]
    if len(parts) >= 2:
        gpu_info = ", ".join(parts)
        gpu_mem_total_mb = int(parts[1])
    print(f"[{t_ts()}] GPU        : {gpu_info or 'NO DETECTADA'}")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print(f"[{t_ts()}] GPU        : nvidia-smi no disponible")

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024 ** 3)
    print(f"[{t_ts()}] RAM        : {ram_gb:.1f} GB")
except ImportError:
    ram_gb = 0.0

root_disk = "/content" if IS_COLAB else "/kaggle" if IS_KAGGLE else "."
disk_free_gb = shutil.disk_usage(root_disk).free / (1024 ** 3)
print(f"[{t_ts()}] Disco libre: {disk_free_gb:.1f} GB en {root_disk}")

# Recomendación de batch en función de GPU mem total
if gpu_mem_total_mb >= 15000:
    SUGGESTED_BATCH = 32
elif gpu_mem_total_mb >= 11000:
    SUGGESTED_BATCH = 24
elif gpu_mem_total_mb >= 8000:
    SUGGESTED_BATCH = 16
else:
    SUGGESTED_BATCH = 8

print(f"\n  Batch sugerido @ imgsz=416 = {SUGGESTED_BATCH} "
      f"(basado en {gpu_mem_total_mb} MB VRAM)")

if not gpu_info:
    print("\n  ⚠️  Sin GPU activa. Cambia a runtime con GPU antes de continuar.")
if disk_free_gb < 8:
    print(f"\n  ⚠️  Solo {disk_free_gb:.1f} GB libres; mosaic+mixup cache puede llenar el disco.")


## 1. Configuración + secrets

Cascada de obtención de `ROBOFLOW_API_KEY` (env → Colab Secrets → Kaggle Secrets → getpass).

In [ ]:
from getpass import getpass

WORKSPACE          = "embebidos3"
PROJECT            = "waste-3class-lwld8"
VERSION            = 1            # Version 1-B: Fit-black 416x416, export yolov8
EPOCHS             = 100
IMGSZ              = 416
BATCH              = SUGGESTED_BATCH    # autoscale por GPU mem
PATIENCE           = 20
SEED               = 42
USE_WANDB          = False        # poner True si tienes cuenta y WANDB_API_KEY
HEARTBEAT_SECS     = 30

def get_secret(name: str, prompt: str | None = None) -> str:
    if name in os.environ and os.environ[name]:
        print(f"  [{t_ts()}] {name} <- env var"); return os.environ[name]
    if IS_COLAB:
        try:
            from google.colab import userdata
            v = userdata.get(name)
            if v:
                os.environ[name] = v
                print(f"  [{t_ts()}] {name} <- Colab Secrets"); return v
        except Exception:
            pass
    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            v = UserSecretsClient().get_secret(name)
            if v:
                os.environ[name] = v
                print(f"  [{t_ts()}] {name} <- Kaggle Secrets"); return v
        except Exception:
            pass
    v = getpass(prompt or f"Pega {name}: ")
    os.environ[name] = v
    return v

banner("1. Secrets")
ROBOFLOW_API_KEY = get_secret("ROBOFLOW_API_KEY", "Pega tu Roboflow API Key: ")
if USE_WANDB:
    get_secret("WANDB_API_KEY", "Pega tu W&B API Key: ")

print(f"\nResumen del experimento:")
for k, v in {
    "workspace": WORKSPACE, "project": PROJECT, "version": VERSION,
    "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
    "patience": PATIENCE, "seed": SEED, "use_wandb": USE_WANDB,
    "heartbeat_secs": HEARTBEAT_SECS,
}.items():
    print(f"  {k:18s}= {v}")


## 2. Persistencia

Drive en Colab / `/kaggle/working` en Kaggle. `PERSIST_ROOT` sobrevive a desconexiones.

In [ ]:
banner("2. Persistencia")

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PERSIST_ROOT = Path("/content/drive/MyDrive/embebidos-3/track_b")
    WORK_DIR     = Path("/content/track_b")
elif IS_KAGGLE:
    PERSIST_ROOT = Path("/kaggle/working/track_b")
    WORK_DIR     = Path("/kaggle/working/track_b_scratch")
else:
    PERSIST_ROOT = Path.cwd() / "out_track_b"
    WORK_DIR     = Path.cwd() / "scratch_track_b"

PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATASET_DIR = WORK_DIR / "ds_yolo"
RUNS_DIR    = WORK_DIR / "runs"
for d in (DATASET_DIR, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Resume from previous run si existe en Drive
prev_runs = PERSIST_ROOT / "runs"
if prev_runs.exists() and any(prev_runs.iterdir()):
    print(f"  [{t_ts()}] Copio runs previos {prev_runs} -> {RUNS_DIR}")
    shutil.copytree(prev_runs, RUNS_DIR, dirs_exist_ok=True)

print(f"\n  WORK_DIR     : {WORK_DIR}")
print(f"  PERSIST_ROOT : {PERSIST_ROOT}")


## 3. Dependencias

Stack training pinneado: `ultralytics>=8.4.46,<8.5` (PR #24028 INT8 calib fix), `onnxsim>=0.6.2` (wheels Py 3.12 desde 0.5.0 — versiones 0.4.x no compilan), `onnx>=1.16,<1.18` (IR compatible con onnxsim). Set `DATASET_DIRECTORY` env var **antes** de instanciar `Roboflow()` para evitar bug `location` (issue roboflow-python #240).

In [ ]:
banner("3. Dependencias")
t0 = time.time()

# === Stack de training (Kaggle/Colab) vs runtime target (Jetson Nano JetPack 4.6.1) ===
#
# RUNTIME TARGET (inmutable, Jetson Nano B01):
#   Python 3.6.9, TRT 8.2.1, CUDA 10.2, cuDNN 8.2.1, OpenCV 4.1.1.
#   GPU Maxwell 128 CUDA cores SIN tensor cores INT8 - FP16 via DP4A.
#   ONNX opset max leible TRT 8.2: 13 (usamos 11 conservador).
#
# STACK TRAINING (decision 2026-05-12, revisada Ronda 3, ver
# investigaciones/2026-05-12/2026-05-12-compatibilidad-notebooks-training.md):
#   Kaggle GPU v168 (mar 2026): Python 3.12, PyTorch 2.10.0+cu128, NumPy 2.4.x default,
#   CUDA host 12.8 (release Kaggle/docker-python v168, marzo 2026).
#   Ultralytics >=8.4.46,<8.5: PR #24028 (INT8 calib no-square) merged en v8.4.31
#   (2026-03-28). v8.4.48 (2026-05-08) es latest. PR #23808 anade safer opset cap
#   para Torch 2.9+ exports en exporter interno.
#   onnxslim >=0.1.82: Ultralytics 8.3+ MIGRO de onnxsim a onnxslim como
#   simplificador interno (verificado leyendo source de exporter.py rama main,
#   2026-05-12). El flag `simplify=True` llama onnxslim, NO onnxsim.
#   NumPy <2.0 PINEADO EXPLICITO antes de ultralytics: Kaggle 2026 default es
#   NumPy 2.4.x; aunque ultralytics declara numpy<2.0 en pyproject.toml,
#   issue #22346 confirma que en Kaggle deps transitivas pueden dejar 2.x
#   instalado y causar "numpy.core.multiarray failed to import" en yolo train.
#
# Roboflow `location` arg bug (cell-10 maneja la cascada de busqueda):
#   v1.3.9 (2026-05-07, SHA 1e4cbc04) SIN fix para el bug (verificado leyendo
#   source de version.py y dataset.py). Releases 1.3.7-1.3.9 enfocados en
#   soft-delete/device-management. Workaround cascada cell-10 sigue obligatorio.
#   NO seteamos DATASET_DIRECTORY env var aqui (genera conflicto con `location`
#   arg y deja ambas ubicaciones vacias).

def run_pip(args: list[str]) -> None:
    print(f"  [{t_ts()}] pip install {' '.join(args)}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

run_pip(["--upgrade", "pip"])

# PASO CRITICO: pinear NumPy <2.0 ANTES de instalar ultralytics.
# Kaggle 2026 trae NumPy 2.4.x default; instalar ultralytics despues respeta el pin
# porque pip resuelve antes de upgradear deps transitivas. Si se pone despues,
# alguna dep transitiva puede ya haber upgradeado numpy a 2.x.
print(f"  [{t_ts()}] Pin defensivo NumPy <2 (Kaggle 2026 default NumPy 2.4.x rompe ultralytics)")
run_pip(["'numpy<2.0'"])

deps = [
    "ultralytics>=8.4.46,<8.5",       # PR #24028 INT8 calib fix (en 8.4.31). Latest 8.4.48.
    "onnx>=1.16,<1.18",               # IR version compatible
    "onnxruntime>=1.18,<1.21",
    "onnxslim>=0.1.82",               # Simplificador real de Ultralytics 8.3+ (NO onnxsim)
    "roboflow>=1.3.6,<1.4",           # Workaround `location` cascada en cell-10
    "tqdm",
]
if USE_WANDB:
    deps.append("wandb")

for pkg in deps:
    try:
        run_pip([pkg])
    except subprocess.CalledProcessError as e:
        print(f"  ! Fallo instalando {pkg}: {e}")

# Defensivo: si DATASET_DIRECTORY quedo set en este kernel (rerun), removerlo
if "DATASET_DIRECTORY" in os.environ:
    print(f"  [{t_ts()}] Removiendo DATASET_DIRECTORY heredado: {os.environ['DATASET_DIRECTORY']}")
    del os.environ["DATASET_DIRECTORY"]

print(f"\n  [{t_ts()}] Listas en {time.time()-t0:.1f}s")

# Smoke check de versiones esperadas
import importlib
expected = {
    "torch": "2.",          # cualquier 2.x (Kaggle 2026: 2.10.x)
    "numpy": "1.",          # CRITICO: debe ser 1.26.x, NO 2.x
    "ultralytics": "8.4.",
    "onnx": "1.",
    "onnxruntime": "1.",
    "onnxslim": "0.1.",     # Simplificador real (no onnxsim)
    "roboflow": "1.3.",
}
print(f"\n  Versiones instaladas:")
for mod, prefix in expected.items():
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        ok = "OK" if str(ver).startswith(prefix) else "WARN"
        print(f"    {ok} {mod:18s}{ver}  (esperado {prefix}*)")
    except ImportError as e:
        print(f"    FAIL {mod}: {e}")

# Validacion dura: NumPy debe ser <2.0 — si no, abortar con mensaje claro
import numpy as _np
if int(_np.__version__.split(".")[0]) >= 2:
    raise RuntimeError(
        f"NumPy {_np.__version__} >= 2.0 instalado. Esto rompe ultralytics + onnx en Kaggle "
        f"(issue #22346). Causa probable: alguna dep transitiva ignoro el pin. "
        f"Workaround: !pip install 'numpy<2.0' --force-reinstall y reinicia kernel."
    )

# Confirmacion critica: PyTorch CUDA disponible
try:
    import torch
    print(f"\n  PyTorch {torch.__version__} CUDA: {torch.cuda.is_available()}  "
          f"(device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'n/a'})")
except Exception as e:
    print(f"  FAIL PyTorch import: {e}")


## 4. Descarga del dataset Roboflow + validación de clases

Skip si ya existe. Valida `nc=3` y `names=[glass, paper, plastic]` para detectar la regresión del bug roboflow-python#88 (clases fantasma).

In [ ]:
banner("4. Dataset Roboflow")

import yaml
import zipfile

def _data_yaml_classes(path: Path) -> set[str] | None:
    """Parse data.yaml y devuelve set de clases lowercase, o None si no parsea."""
    try:
        with open(path, encoding="utf-8", errors="ignore") as f:
            d = yaml.safe_load(f)
        if not isinstance(d, dict):
            return None
        names = d.get("names")
        if isinstance(names, dict):
            names = list(names.values())
        if not isinstance(names, list):
            return None
        return {str(n).strip().lower() for n in names}
    except (yaml.YAMLError, OSError, UnicodeError):
        return None

def find_all_data_yamls() -> list[tuple[Path, set[str] | None]]:
    """Busca TODOS los data.yaml en ubicaciones probables (Kaggle + Colab + local)."""
    roots: set[Path] = {
        Path.cwd(), WORK_DIR, WORK_DIR.parent, Path.home(),
    }
    extra_paths = [
        os.environ.get("DATASET_DIRECTORY", ""),
        "/kaggle/working", "/kaggle/working/datasets",
        "/content", "/content/datasets",
    ]
    for p in extra_paths:
        if p:
            roots.add(Path(p))

    found: list[tuple[Path, set[str] | None]] = []
    seen: set[Path] = set()
    for root in roots:
        if not root.exists():
            continue
        try:
            for cand in root.glob("**/data.yaml"):
                cand_resolved = cand.resolve()
                if cand_resolved in seen:
                    continue
                seen.add(cand_resolved)
                classes = _data_yaml_classes(cand)
                found.append((cand, classes))
        except (PermissionError, OSError):
            continue
    return found

def pick_correct_data_yaml(candidates: list[tuple[Path, set[str] | None]]) -> Path | None:
    """Elige el data.yaml con clases exactas {glass, paper, plastic}; fallback al primero."""
    expected = {"glass", "paper", "plastic"}
    for path, classes in candidates:
        if classes == expected:
            return path
    for path, classes in candidates:
        if classes and len(classes & expected) >= 2:
            return path
    for path, classes in candidates:
        if classes and len(classes) == 3:
            return path
    return candidates[0][0] if candidates else None

def extract_any_zips_in(root: Path) -> int:
    """Si el SDK dejó zips sin extraer, extraerlos. Devuelve cantidad extraída."""
    if not root.exists():
        return 0
    n = 0
    for z in list(root.rglob("*.zip")):
        try:
            with zipfile.ZipFile(z) as zf:
                target = z.parent / z.stem
                target.mkdir(parents=True, exist_ok=True)
                zf.extractall(target)
            print(f"  [{t_ts()}] Zip extraído: {z} → {target}")
            n += 1
        except (zipfile.BadZipFile, OSError) as e:
            print(f"  ! No pude extraer {z}: {e}")
    return n

# CRITICAL: remover DATASET_DIRECTORY si está set — entra en conflicto con `location` arg
# (el SDK puede usar env var aunque pasemos `location`, generando dos ubicaciones distintas
# y dejando ambas vacías o el zip sin extraer).
if "DATASET_DIRECTORY" in os.environ:
    print(f"  [{t_ts()}] Removiendo DATASET_DIRECTORY env var (conflict con location arg)")
    del os.environ["DATASET_DIRECTORY"]

existing = next(iter(DATASET_DIR.glob("**/data.yaml")), None)
ds = None
if existing is None:
    from roboflow import Roboflow

    print(f"  [{t_ts()}] DATASET_DIR : {DATASET_DIR}")
    print(f"  [{t_ts()}] CWD antes   : {Path.cwd()}")

    # Mitigación bug: chdir a DATASET_DIR antes de download
    original_cwd = Path.cwd()
    os.chdir(DATASET_DIR)
    try:
        print(f"  [{t_ts()}] Descargando {WORKSPACE}/{PROJECT}/v{VERSION} (yolov8)...")
        rf = Roboflow(api_key=ROBOFLOW_API_KEY)
        proj = rf.workspace(WORKSPACE).project(PROJECT)
        ds = proj.version(VERSION).download(
            "yolov8", location=str(DATASET_DIR), overwrite=False
        )
        print(f"  [{t_ts()}] ds.location: {ds.location}")
        print(f"  [{t_ts()}] CWD después: {Path.cwd()}")
    finally:
        os.chdir(original_cwd)

    # Extraer zips no extraídos en posibles ubicaciones (defensivo)
    for diag_root in (DATASET_DIR, WORK_DIR, Path.cwd(),
                       Path("/kaggle/working") if Path("/kaggle/working").exists() else None,
                       Path("/content") if Path("/content").exists() else None):
        if diag_root:
            extract_any_zips_in(diag_root)

    # Búsqueda exhaustiva con yaml parse
    print(f"\n  [{t_ts()}] Búsqueda exhaustiva de data.yaml ...")
    all_yamls = find_all_data_yamls()
    print(f"  [{t_ts()}] Encontrados {len(all_yamls)} archivos data.yaml:")
    for path, classes in all_yamls:
        cls_str = sorted(classes) if classes else "?"
        try:
            size = path.stat().st_size
        except OSError:
            size = -1
        print(f"    {path}  ({size} bytes, classes={cls_str})")

    data_yaml_path = pick_correct_data_yaml(all_yamls)

    # Si lo encontró fuera de DATASET_DIR, mover físicamente
    if data_yaml_path is not None and DATASET_DIR not in data_yaml_path.parents:
        src_dir = data_yaml_path.parent
        print(f"\n  [{t_ts()}] Migrando contenido desde {src_dir} → {DATASET_DIR}")
        for item in src_dir.iterdir():
            dst = DATASET_DIR / item.name
            if dst.exists():
                continue
            try:
                if item.is_dir():
                    shutil.copytree(item, dst)
                else:
                    shutil.copy2(item, dst)
            except (OSError, shutil.Error) as e:
                print(f"    ! No pude copiar {item.name}: {e}")
        data_yaml_path = next(iter(DATASET_DIR.glob("**/data.yaml")), None)
else:
    data_yaml_path = existing
    print(f"  [{t_ts()}] Dataset ya presente.")

if data_yaml_path is None:
    print(f"\n  ⚠️  data.yaml NO encontrado tras cascada de búsqueda.")

    # Diagnóstico exhaustivo de todas las ubicaciones probables
    diag_paths = [
        DATASET_DIR, WORK_DIR, Path.cwd(),
        Path("/kaggle/working") if Path("/kaggle/working").exists() else None,
        Path("/kaggle/working/datasets") if Path("/kaggle/working/datasets").exists() else None,
        Path("/content") if Path("/content").exists() else None,
        Path("/content/datasets") if Path("/content/datasets").exists() else None,
    ]
    diag_paths = [p for p in diag_paths if p and p.exists()]
    for diag_root in diag_paths:
        print(f"\n  Contenido recursivo (primeros 40) de {diag_root}:")
        items = sorted(diag_root.rglob("*"))[:40]
        if not items:
            print(f"    (vacío)")
        else:
            for p in items:
                try:
                    rel = p.relative_to(diag_root)
                    kind = "d" if p.is_dir() else "f"
                    size = p.stat().st_size if p.is_file() else ""
                    print(f"    [{kind}] {rel}  {size}")
                except Exception:
                    print(f"    {p}")

    raise FileNotFoundError(
        f"Roboflow no produjo data.yaml. Posible bug roboflow-python: "
        f"location ignored o extracción del zip falló. Diagnóstico arriba; "
        f"verifica ROBOFLOW_API_KEY y permisos del workspace."
    )

DATA_YAML = data_yaml_path
print(f"\n  [{t_ts()}] data.yaml resuelto: {DATA_YAML}")

with open(DATA_YAML) as f:
    meta = yaml.safe_load(f)

print(f"\n  Metadata data.yaml:")
for k, v in meta.items():
    print(f"    {k}: {v}")

# Normalizar nombres a lowercase para validación (Roboflow puede exportar capitalized)
names_lower = [str(n).strip().lower() for n in meta.get("names", [])]
expected = {"glass", "paper", "plastic"}
got = set(names_lower)

# Gotcha bug roboflow-python#88: clases fantasma post-Modify Classes
if meta.get("nc") != 3 or got != expected:
    print(f"\n  ⚠️  data.yaml tiene clases inesperadas:")
    print(f"      nc actual = {meta.get('nc')}, esperado 3")
    print(f"      names actuales = {meta.get('names')}, esperado lowercase {sorted(expected)}")
    print(f"      Posible bug roboflow-python#88 (clases fantasma).")
    if expected.issubset(got):
        print(f"      Las 3 clases esperadas SÍ están presentes; continuamos.")
    else:
        raise AssertionError(
            f"Faltan clases esperadas: {expected - got}. "
            f"Re-verifica Roboflow Version 1-B."
        )
else:
    print(f"  ✓ Clases validadas: {sorted(got)}")

# Conteo de imágenes por split (paths son relativos a data.yaml en Roboflow yolov8 format)
for split in ("train", "val", "test"):
    p = meta.get(split)
    if not p:
        continue
    abs_path = (DATA_YAML.parent / p).resolve()
    if abs_path.is_file():
        n = sum(1 for _ in abs_path.open())
    elif abs_path.is_dir():
        n = sum(1 for _ in abs_path.glob("*.*"))
    else:
        n = 0
        print(f"    ! split '{split}' apunta a path inexistente: {abs_path}")
        continue
    print(f"  {split:6s}: {n} imgs en {abs_path}")


## 5. Pre-flight check antes de training

Libera RAM/GPU caches, valida que YOLO carga el pretrained, imprime el resumen de hiperparámetros.

In [ ]:
banner("5. Pre-flight training")

import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_mb = (torch.cuda.mem_get_info()[0]) / (1024**2)
    print(f"  [{t_ts()}] GPU mem libre: {free_mb:.0f} MB")

from ultralytics import YOLO
model = YOLO("yolov8n.pt")
print(f"  [{t_ts()}] yolov8n.pt cargado")
print(f"    Parámetros : {sum(p.numel() for p in model.model.parameters())/1e6:.2f}M")
print(f"    GFLOPS@416 : ~3.3 (estimado; bench real en Nano)\n")

print(f"  Config training:")
for k, v in {
    "data": str(data_yaml_path), "epochs": EPOCHS, "imgsz": IMGSZ,
    "batch": BATCH, "patience": PATIENCE, "seed": SEED,
    "optimizer": "SGD lr=0.01 mom=0.937 wd=5e-4",
    "augs_online": "mosaic=1.0 close=10 mixup=0.15 fliplr=0.5",
}.items():
    print(f"    {k:18s}= {v}")


## 6. TensorBoard inline + callback de heartbeat

Ultralytics ya muestra una barra por epoch. Le añadimos un **heartbeat custom cada `HEARTBEAT_SECS` segundos** que loggea: epoch, mAP50 actual, GPU mem, RSS, ETA.

In [ ]:
banner("6. Callbacks")

try:
    get_ipython().run_line_magic("load_ext", "tensorboard")
    get_ipython().run_line_magic("tensorboard", f"--logdir {RUNS_DIR}")
    print(f"  [{t_ts()}] TensorBoard cargado en {RUNS_DIR}")
except Exception as e:
    print(f"  ! TensorBoard inline no disponible: {e}")

# Heartbeat callback
import threading
hb_state = {"last_seen": time.time(), "dead": False,
            "epoch": 0, "metrics": {}}

def _gpu_used_mb() -> int:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=3,
        )
        return int(out.stdout.strip())
    except Exception:
        return 0

def hb_loop() -> None:
    started = time.time()
    while not hb_state["dead"]:
        time.sleep(HEARTBEAT_SECS)
        if hb_state["dead"]: break
        rss_mb = 0.0
        try:
            import psutil
            rss_mb = psutil.Process().memory_info().rss / (1024**2)
        except Exception:
            pass
        gpu_mb = _gpu_used_mb()
        elapsed_min = (time.time() - started) / 60
        epoch = hb_state["epoch"]
        eta = "n/a"
        if epoch > 0:
            mins_per_epoch = elapsed_min / epoch
            eta = f"{mins_per_epoch * (EPOCHS - epoch):.1f} min"
        last_metrics = hb_state["metrics"]
        map50 = last_metrics.get("metrics/mAP50(B)", float("nan"))
        loss = last_metrics.get("train/box_loss", float("nan"))
        print(f"  [HEARTBEAT {t_ts()}] epoch {epoch}/{EPOCHS}  "
              f"mAP50={map50:.3f}  box_loss={loss:.3f}  "
              f"GPU={gpu_mb}MB  RSS={rss_mb:.0f}MB  ETA~{eta}",
              flush=True)

def on_train_epoch_end(trainer):
    hb_state["epoch"] = trainer.epoch + 1
    hb_state["metrics"] = {**(trainer.metrics or {}),
                            "train/box_loss": float(trainer.loss_items[0])
                            if trainer.loss_items is not None else float("nan")}

def on_train_end(trainer):
    hb_state["dead"] = True

model.add_callback("on_train_epoch_end", on_train_epoch_end)
model.add_callback("on_train_end", on_train_end)
print(f"  [{t_ts()}] Callbacks registrados.")


## 7. Training con monitoreo en vivo

Lanza el heartbeat en hilo paralelo a `model.train(...)`. Si la celda no muestra heartbeat por 2 minutos, presume cuelgue y revisa logs.

In [ ]:
banner("7. Training")

hb_state["dead"] = False
hb_thread = threading.Thread(target=hb_loop, daemon=True)
hb_thread.start()

t_train = time.time()
try:
    results = model.train(
        data=str(data_yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        project=str(RUNS_DIR),
        name="yolov8n_waste_v1",
        exist_ok=True,
        optimizer="SGD",
        lr0=0.01, momentum=0.937, weight_decay=0.0005,
        warmup_epochs=3, cos_lr=True,
        patience=PATIENCE,
        # Augmentations online (complementan Roboflow Aug 3x):
        mosaic=1.0, close_mosaic=10,
        mixup=0.15,
        flipud=0.0, fliplr=0.5,
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
        translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
        degrees=0.0,
        seed=SEED, deterministic=True,
        plots=True, verbose=True,
    )
finally:
    hb_state["dead"] = True
    hb_thread.join(timeout=2)

print(f"\n  [{t_ts()}] Training terminó en {(time.time()-t_train)/60:.1f} min")

# Path del best.pt
run_dir = Path(model.trainer.save_dir)
best_pt = run_dir / "weights" / "best.pt"
print(f"  best.pt: {best_pt}  ({best_pt.stat().st_size/(1024**2):.1f} MB)")


## 8. Evaluación en el split `test`

Reporta mAP@50 global + por clase. Estructurado en JSON para el informe IEEE.

In [ ]:
banner("8. Evaluación test")

best_model = YOLO(str(best_pt))
metrics = best_model.val(data=str(data_yaml_path), split="test", imgsz=IMGSZ, verbose=True)

eval_summary = {
    "mAP50_global"   : float(metrics.box.map50),
    "mAP50_95_global": float(metrics.box.map),
    "ap50_per_class" : {},
}
for i, name in enumerate(meta["names"]):
    eval_summary["ap50_per_class"][name] = float(metrics.box.ap50[i])

print(f"\n  Resultados sobre TEST:")
print(f"    mAP@50    : {eval_summary['mAP50_global']:.4f}")
print(f"    mAP@50:95 : {eval_summary['mAP50_95_global']:.4f}")
for name, ap in eval_summary["ap50_per_class"].items():
    print(f"    AP@50 {name:8s}: {ap:.4f}")


## 9. Export ONNX (opset 11) + validación

Export + `onnx.checker.check_model` para validar que TRT 8.0/8.2 lo va a aceptar.

In [ ]:
banner("9. Export ONNX")

# CRITICO: opset=11 explicito.
# Ultralytics 8.4.x con torch 2.9+ usa best_onnx_opset() que retorna opset 20-22
# por default (legacy TorchScript cap), pero TRT 8.2.1 del JetPack 4.6.1 solo lee
# hasta opset 13. Usamos 11 conservador. Si se omite opset=, Ronda 3 confirma
# que se genera opset 20+ -> TRT 8.2 falla con "Unsupported ONNX data type" o
# "Gather rank-0 not supported" (issue NVIDIA/TensorRT #4383).
# nms=False: NMS se hace en CPU NumPy en Nano (EfficientNMS_TRT roto Maxwell sm_53,
# NVIDIA/TensorRT issue #1538 desde 2021, no fixed para Maxwell en TRT 8.x).
# simplify=True: invoca onnxslim (NO onnxsim) en Ultralytics 8.3+ (cambio
# documentado leyendo source de exporter.py rama main 2026-05-12).
onnx_path_str = best_model.export(
    format="onnx",
    opset=11,           # OBLIGATORIO TRT 8.0/8.2 (no lee > 13). Sin esto: opset 20+.
    imgsz=IMGSZ,
    simplify=True,      # onnxslim integrado (Ultralytics 8.3+ migro de onnxsim)
    dynamic=False,      # shapes fijas para TRT engine deterministico
    half=False,         # FP16 se aplica en trtexec --fp16 en el Nano
    nms=False,          # NMS en CPU NumPy (EfficientNMS_TRT roto Maxwell)
    device="cpu",       # CUDA host irrelevante para .onnx, evita confusion
)
ONNX_PATH = Path(onnx_path_str)
print(f"  [{t_ts()}] ONNX exportado: {ONNX_PATH}  ({ONNX_PATH.stat().st_size/(1024**2):.1f} MB)")

# Validacion structural ONNX
import onnx
onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
print(f"  OK onnx.checker passed")
print(f"  Opset                : {onnx_model.opset_import[0].version}")
print(f"  IR version           : {onnx_model.ir_version}")
print(f"  Input shape          : {[d.dim_value for d in onnx_model.graph.input[0].type.tensor_type.shape.dim]}")
print(f"  Outputs              : {len(onnx_model.graph.output)}")

# Validaciones criticas pre-deploy:
assert onnx_model.opset_import[0].version == 11, \
    f"Opset esperado 11, encontrado {onnx_model.opset_import[0].version}. TRT 8.2 fallara."
assert onnx_model.ir_version <= 10, \
    f"IR version {onnx_model.ir_version} > 10 puede romper onnxslim/onnxsim (issue #367)."
# Confirmacion NMS ausente: con nms=False, outputs son raw [boxes_scores] (1 tensor)
# en YOLOv8 detect. Con nms=True serian 4 tensores [boxes, classes, scores, num_det].
print(f"\n  OK Validaciones pre-deploy:")
print(f"    opset == 11       : {onnx_model.opset_import[0].version == 11}")
print(f"    ir_version <= 10  : {onnx_model.ir_version <= 10}")
print(f"    nms ausente       : {len(onnx_model.graph.output) <= 2} (outputs={len(onnx_model.graph.output)})")

# Validacion adicional Ronda 3: detectar ops problematicos para TRT 8.2.1.
# Reciprocal, NonZero, RoiAlign no son soportados en TRT 8.2 (onnx-tensorrt
# release/8.2-GA operators.md). En YOLOv8n FP32 estatico sin NMS estos no deberian
# aparecer, pero validamos para no descubrirlo despues en el Nano.
PROBLEMATIC_OPS = {"Reciprocal", "NonZero", "RoiAlign", "QLinearConv", "QLinearMatMul"}
ops_used = {node.op_type for node in onnx_model.graph.node}
problematic = ops_used & PROBLEMATIC_OPS
if problematic:
    print(f"\n  WARN Ops problematicos para TRT 8.2.1 detectados: {sorted(problematic)}")
    print(f"        Revisar grafo antes de trtexec en Nano.")
else:
    print(f"    ops_problematicos : ninguno (ops totales: {len(ops_used)})")


## 10. Smoke test con ONNXRuntime

Carga el ONNX en CPU, corre inferencia sobre una imagen del test, mide latencia. NO es proxy de Jetson FP16, solo verifica que el grafo es válido.

In [ ]:
banner("10. Smoke test ONNX")
import onnxruntime as ort
import numpy as np
from PIL import Image

# Encuentra una imagen del test
test_path = (data_yaml_path.parent / meta.get("test", "test/images")).resolve()
if test_path.is_file():
    with test_path.open() as f:
        sample_img = Path(f.readline().strip())
else:
    sample_img = next(test_path.glob("*.jpg"), None) or next(test_path.glob("*.png"))
print(f"  Imagen de prueba: {sample_img}")

img = Image.open(sample_img).convert("RGB").resize((IMGSZ, IMGSZ), Image.BILINEAR)
arr = np.asarray(img, dtype=np.float32).transpose(2, 0, 1) / 255.0  # CHW
arr = np.expand_dims(arr, 0)

sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name

# Warm-up + 5 runs
sess.run(None, {input_name: arr})
lats = []
for _ in range(5):
    t0 = time.perf_counter()
    out = sess.run(None, {input_name: arr})
    lats.append((time.perf_counter() - t0) * 1000)

print(f"  Output shape  : {[o.shape for o in out]}")
print(f"  Latencia CPU  : min={min(lats):.1f}ms p50={sorted(lats)[2]:.1f}ms max={max(lats):.1f}ms")
print(f"  [{t_ts()}] Smoke test OK")


## 11. Manifest JSON + persistir a Drive

Genera `manifest_track_b.json` con todo lo reproducible y copia a `PERSIST_ROOT`.

In [ ]:
banner("11. Manifest + persistencia")
import hashlib

md5 = hashlib.md5(ONNX_PATH.read_bytes()).hexdigest()
sha256 = hashlib.sha256(ONNX_PATH.read_bytes()).hexdigest()

manifest = {
    "track": "B",
    "model": "yolov8n -> onnx_opset11 -> (trt_fp16 en Nano)",
    "platform_train": PLATFORM,
    "stack_train": {
        "python": sys.version.split()[0],
        "pytorch": __import__("torch").__version__,
        "ultralytics": __import__("ultralytics").__version__,
        "onnx": __import__("onnx").__version__,
        "onnxsim": __import__("onnxsim").__version__,
    },
    "stack_runtime_target": {
        "device": "Jetson Nano B01",
        "jetpack": "4.6.1",
        "tensorrt": "8.2.1",
        "cuda": "10.2",
        "inference": "TRT FP16 engine + NMS CPU NumPy",
    },
    "roboflow": {"workspace": WORKSPACE, "project": PROJECT,
                 "version": VERSION, "format": "yolov8",
                 "resize": "fit-black 416x416"},
    "training": {
        "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
        "optimizer": "SGD lr=0.01 mom=0.937 wd=5e-4",
        "augs_online": {"mosaic": 1.0, "close_mosaic": 10,
                          "mixup": 0.15, "fliplr": 0.5},
        "patience": PATIENCE, "seed": SEED,
        "wallclock_minutes": (time.time() - t_train) / 60,
    },
    "artifact": {
        "filename": ONNX_PATH.name,
        "size_mb": round(ONNX_PATH.stat().st_size / (1024**2), 2),
        "md5": md5, "sha256": sha256,
        "opset": onnx_model.opset_import[0].version,
        "ir_version": onnx_model.ir_version,
        "nms_embedded": False,
    },
    "metrics_test": eval_summary,
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
}

MANIFEST_PATH = WORK_DIR / "manifest_track_b.json"
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

print(f"\n  [{t_ts()}] Persistiendo a {PERSIST_ROOT}...")
shutil.copy(ONNX_PATH, PERSIST_ROOT / ONNX_PATH.name)
shutil.copy(MANIFEST_PATH, PERSIST_ROOT / MANIFEST_PATH.name)
shutil.copy(best_pt, PERSIST_ROOT / best_pt.name)
shutil.copytree(run_dir, PERSIST_ROOT / "runs" / run_dir.name, dirs_exist_ok=True)

print(f"\n  ✓ {ONNX_PATH.name} listo para scp al Jetson Nano.")
print(f"    En el Nano:")
print(f"      sudo systemctl stop lightdm")
print(f"      export PATH=$PATH:/usr/src/tensorrt/bin")
print(f"      trtexec --onnx={ONNX_PATH.name} \\")
print(f"              --saveEngine=yolov8n_waste_fp16.engine \\")
print(f"              --fp16 --workspace=1024 --verbose")
print(f"    Tiempo estimado: 15-45 min en Nano 4 GB.")
print(f"    Post-build: NMS en CPU NumPy con torchvision.ops.nms o cv2.dnn.NMSBoxes.")
